# Multimodal feature fusion

Intermediate fusion combines the learned feature vectors from the three trained modality models. Each modality vector is concatenated, then a multilayer perceptron (MLP) learns the final clickbait prediction. The notebook selects the fusion head using validation loss, reports validation accuracy, F1, and a confusion matrix, then writes test predictions.

**Required inputs:** each existing modality model must export one CSV per split under `artifacts/fusion/`. CSVs must contain `video_id` and numeric feature columns; train and validation files must also contain `final_label` (0 or 1). Feature vectors must come from the trained model's representation before its final classifier. Keep the same split and video IDs across all modalities. This notebook does not tune on the test split."

## 1. Imports and reproducibility

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss, classification_report, ConfusionMatrixDisplay
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
LABEL = "final_label"
ID = "video_id"
MODALITIES = ("numeric", "thumbnail", "text")
SPLITS = ("train", "validation", "test")


## 2. Configure artifact locations

Run from the repository root or from `fusion/`. Adjust `ARTIFACT_DIR` if your model notebooks save their exports elsewhere. Expected filenames are `train_numeric_features.csv`, `train_thumbnail_features.csv`, and `train_text_features.csv` (and the equivalent validation/test files).

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir() and (ROOT.parent / "data" / "processed").is_dir():
    ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts" / "fusion"
OUTPUT_DIR = ARTIFACT_DIR / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILES = {
    split: {modality: ARTIFACT_DIR / f"{split}_{modality}_features.csv"
            for modality in MODALITIES}
    for split in SPLITS
}
for split, paths in FEATURE_FILES.items():
    for modality, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split}/{modality} features: {path}. "
                "Export a feature CSV from each trained modality model first."
            )
print("Feature artifacts found for all splits and modalities.")


## 3. Load, validate, and align modality features

The loader checks unique video IDs, numeric finite features, binary labels, and identical video coverage across modalities. It prefixes feature names with the modality to make concatenated columns unambiguous.

In [ ]:
def load_split(split):
    require_label = split != "test"
    frames = {}
    for modality in MODALITIES:
        path = FEATURE_FILES[split][modality]
        frame = pd.read_csv(path)
        required = {ID} | ({LABEL} if require_label else set())
        missing = required - set(frame.columns)
        if missing:
            raise KeyError(f"{path} is missing required columns: {sorted(missing)}")
        if frame[ID].isna().any() or frame[ID].duplicated().any():
            raise ValueError(f"{path} has missing or duplicate video_id values")
        feature_cols = [c for c in frame.columns if c not in {ID, LABEL}]
        if not feature_cols:
            raise ValueError(f"{path} has no feature columns")
        frame[feature_cols] = frame[feature_cols].apply(pd.to_numeric, errors="coerce")
        if frame[feature_cols].isna().any().any():
            raise ValueError(f"{path} contains missing or non-numeric features")
        values = frame[feature_cols].to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"{path} contains infinite features")
        frame = frame.rename(columns={c: f"{modality}__{c}" for c in feature_cols})
        frames[modality] = frame

    reference_ids = set(frames[MODALITIES[0]][ID])
    for modality, frame in frames.items():
        if set(frame[ID]) != reference_ids:
            raise ValueError(f"{split}: video_id coverage differs for modality {modality}")
    if require_label:
        for modality, frame in frames.items():
            frame[LABEL] = pd.to_numeric(frame[LABEL], errors="raise").astype(int)
            if not set(frame[LABEL].unique()).issubset({0, 1}):
                raise ValueError(f"{split}/{modality}: labels must be 0 or 1")
        label_maps = [frames[m].set_index(ID)[LABEL].sort_index() for m in MODALITIES]
        if not all(label_maps[0].equals(other) for other in label_maps[1:]):
            raise ValueError(f"{split}: labels disagree across modality exports")

    merged = frames[MODALITIES[0]].set_index(ID)
    for modality in MODALITIES[1:]:
        other = frames[modality].set_index(ID).drop(columns=[LABEL], errors="ignore")
        merged = merged.join(other, how="inner", validate="one_to_one")
    return merged.sort_index().reset_index()

train_df = load_split("train")
validation_df = load_split("validation")
test_df = load_split("test")
if set(train_df[ID]) & set(validation_df[ID]):
    raise ValueError("Train and validation video IDs overlap; check the split exports.")
if set(train_df[ID]) & set(test_df[ID]) or set(validation_df[ID]) & set(test_df[ID]):
    raise ValueError("Test video IDs overlap with train or validation.")
FEATURE_COLUMNS = [c for c in train_df.columns if c not in {ID, LABEL}]
if set(FEATURE_COLUMNS) != {c for c in validation_df.columns if c not in {ID, LABEL}}:
    raise ValueError("Train and validation feature dimensions/names differ.")
if set(FEATURE_COLUMNS) != {c for c in test_df.columns if c not in {ID, LABEL}}:
    raise ValueError("Train and test feature dimensions/names differ.")
print(f"Aligned rows — train: {len(train_df):,}; validation: {len(validation_df):,}; test: {len(test_df):,}")
print(f"Concatenated feature dimensions: {len(FEATURE_COLUMNS):,}")


## 4. Scale concatenated features

The scaler is fit only on the training features to avoid leaking validation or test information.

In [ ]:
X_train_raw = train_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y_train = train_df[LABEL].to_numpy(dtype=np.int64)
X_val_raw = validation_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y_val = validation_df[LABEL].to_numpy(dtype=np.int64)
X_test_raw = test_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)
print("Feature matrices:", X_train.shape, X_val.shape, X_test.shape)


## 5. Define and train the fusion MLP

The primary architecture is concatenation followed by a two-layer MLP. Each epoch is evaluated on validation log loss; the best weights are restored after patience-based early stopping.

In [ ]:
fusion_model = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=1,
    warm_start=True,
    random_state=SEED,
)

best_val_loss = np.inf
best_weights = None
best_epoch = 0
patience = 8
stale_epochs = 0
max_epochs = 100

for epoch in range(1, max_epochs + 1):
    if epoch == 1:
        fusion_model.partial_fit(X_train, y_train, classes=np.array([0, 1]))
    else:
        fusion_model.partial_fit(X_train, y_train)
    val_prob = fusion_model.predict_proba(X_val)[:, 1]
    val_loss = log_loss(y_val, val_prob, labels=[0, 1])
    print(f"Epoch {epoch:03d} | validation log loss: {val_loss:.5f}")
    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_epoch = epoch
        best_weights = (deepcopy(fusion_model.coefs_), deepcopy(fusion_model.intercepts_))
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= patience:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

if best_weights is None:
    raise RuntimeError("Fusion model did not produce a valid checkpoint.")
fusion_model.coefs_, fusion_model.intercepts_ = best_weights
print(f"Restored best validation checkpoint from epoch {best_epoch} (loss={best_val_loss:.5f}).")


## 6. Validation metrics and confusion matrix

This is the final validation check for the fusion model. F1 is reported as both binary (positive class 1) and macro averaged.

In [ ]:
val_prob = fusion_model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)
val_accuracy = accuracy_score(y_val, val_pred)
val_f1_binary = f1_score(y_val, val_pred, average="binary", zero_division=0)
val_f1_macro = f1_score(y_val, val_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_val, val_pred, labels=[0, 1])

print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation F1 (positive class): {val_f1_binary:.4f}")
print(f"Validation F1 (macro): {val_f1_macro:.4f}")
print("\nClassification report:")
print(classification_report(y_val, val_pred, labels=[0, 1], target_names=["not clickbait", "clickbait"], zero_division=0))
display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not clickbait", "clickbait"]).plot(cmap="Blues", values_format="d")
plt.title("Validation confusion matrix — concatenation + MLP")
plt.tight_layout()
plt.show()


## 7. Produce final test predictions

After validation review, apply the selected fusion model and training-fitted scaler to the untouched test features. If test labels are present, they are intentionally ignored here; the reported model-selection metrics remain validation-only.

In [ ]:
test_prob = fusion_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
predictions = pd.DataFrame({
    ID: test_df[ID],
    "clickbait_probability": test_prob,
    "final_prediction": test_pred,
})
prediction_path = OUTPUT_DIR / "test_fusion_predictions.csv"
predictions.to_csv(prediction_path, index=False)
print(f"Saved {len(predictions):,} final predictions to {prediction_path}")
display(predictions.head())


## 8. Save the validation summary

Save the validation metrics alongside the final predictions so the fusion run is reproducible and reviewable.

In [ ]:
metrics = pd.DataFrame([{
    "model": "concatenation_mlp",
    "best_epoch": best_epoch,
    "validation_accuracy": val_accuracy,
    "validation_f1_binary": val_f1_binary,
    "validation_f1_macro": val_f1_macro,
    "validation_log_loss": best_val_loss,
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "feature_dimensions": len(FEATURE_COLUMNS),
}])
metrics_path = OUTPUT_DIR / "validation_metrics.csv"
metrics.to_csv(metrics_path, index=False)
print(f"Saved validation metrics to {metrics_path}")
display(metrics)
